# 6_CLUSTERING — Geophysical Feature Clustering v1.0

Fits **K-Means** (K = 3, 4, 5) on the **combined global IHFC reference set** using all
`obsmodel` observables.  The same cluster centroids are then applied to both the
**Greenland** and **Antarctica** prediction grids, so cluster labels are directly
comparable across the two domains.

| Section | Description | Est. time |
|---------|-------------|-----------|
| 1 | Imports & config | — |
| 2 | Load reference data & scale | < 1 min |
| 3 | Fit K-Means (K = 3, 4, 5) on reference | < 1 min |
| 4 | Cluster diagnostics (inertia, silhouette, Davies-Bouldin) | < 1 min |
| 5 | Apply to Greenland & Antarctica grids | ~ 1–2 min |
| 6 | Visualise cluster maps | ~ 1 min |
| 7 | Save cluster label arrays | — |

**Prerequisites:** `1_Import.ipynb` has been run and
`data/IHFCobs.parquet`, `output/targets/greenland_grid.parquet`,
and `output/targets/antarctica_grid.parquet` exist.


## 0. Toggles

In [1]:
# Set True to re-fit; False to reload saved models from disk
RUN_CLUSTERING = True   # False → load output/clustering/kmeans_k*.pkl

# K values to test
K_VALUES = [3, 4, 5]

# Paths to target-grid parquets (adjust if your naming differs)
GREENLAND_PARQUET  = "output/targets/greenland_grid.parquet"
ANTARCTICA_PARQUET = "output/targets/antarctica_grid.parquet"


## 1. Imports & setup

In [2]:
import sys, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

sys.path.insert(0, ".")
from config import *

warnings.filterwarnings("ignore")

RANDOM_SEED  = 42
QCLIPMIN     = 0.0
CLUSTER_DIR  = Path("output/clustering")
CLUSTER_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR      = Path("output/figures/6_CLUSTERING")
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"obsmodel features : {len(obs_model)}")
print(f"K values          : {K_VALUES}")
print(obs_model)


✓ config.py v4.0 | obs_model: 21 | obs_sweep: 22 | obs: 36
obsmodel features : 21
K values          : [3, 4, 5]
['MOHO', 'MOHO_GRAV', 'DEM', 'LAB', 'FREE_AIR', 'BOUGUER', 'SI', 'GEOID', 'REVEAL_S80', 'REVEAL_S90', 'REVEAL_S70', 'REVEAL_S100', 'REVEAL_VP60VS70', 'REVEAL_VP90VS60', 'REVEAL_VP50VS80', 'LITH_RHO', 'CRUST_RHO', 'MAG_SEIS_MOHO', 'SEDIMENT', 'CTD', 'EMAG2_LOG']


## 2. Load & scale reference data

In [3]:
df_ref = pd.read_parquet(parquet_ref)
print(f"Raw reference rows : {len(df_ref):,}")

# Keep only rows with complete obsmodel columns and valid heat-flow
cols_needed = obs_model + ["q"]

X_ref = df_ref[obs_model].values.astype(np.float32)
print(f"After NaN / clip   : {len(df_ref):,} rows")
print(f"Feature matrix     : {X_ref.shape}")

# Fit scaler on reference — will be reused for both target grids
scaler = StandardScaler().fit(X_ref)
X_ref_sc = scaler.transform(X_ref)
print("StandardScaler fitted on reference.")


Raw reference rows : 30,848
After NaN / clip   : 30,848 rows
Feature matrix     : (30848, 21)
StandardScaler fitted on reference.


## 3. Fit K-Means on the global reference set

A single K-Means model is fitted per K value using **all reference observations**.
Spatial coordinates are **excluded** from clustering so that geographically
separated domains (Greenland, Antarctica) can share the same feature-space
partition without penalising distance.


In [4]:
kmeans_models = {}   # k -> fitted KMeans object
ref_labels    = {}   # k -> cluster label array for reference rows

if RUN_CLUSTERING:
    for k in K_VALUES:
        print(f"Fitting KMeans  K={k} ...", end=" ", flush=True)
        km = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=20, max_iter=500)
        km.fit(X_ref_sc)
        kmeans_models[k] = km
        ref_labels[k]    = km.labels_
        # Save model
        pkl_path = CLUSTER_DIR / f"kmeans_k{k}.pkl"
        with open(pkl_path, "wb") as fp:
            pickle.dump(km, fp)
        print(f"done.  Inertia={km.inertia_:.1f}  Saved → {pkl_path}")
else:
    for k in K_VALUES:
        pkl_path = CLUSTER_DIR / f"kmeans_k{k}.pkl"
        with open(pkl_path, "rb") as fp:
            km = pickle.load(fp)
        kmeans_models[k] = km
        ref_labels[k]    = km.predict(X_ref_sc)
        print(f"Loaded KMeans K={k} from {pkl_path}")


Fitting KMeans  K=3 ... 

ValueError: Input X contains NaN.
KMeans does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

## 4. Cluster diagnostics

In [ ]:
diag_rows = []
for k in K_VALUES:
    km  = kmeans_models[k]
    lbl = ref_labels[k]
    sil = silhouette_score(X_ref_sc, lbl, sample_size=10_000,
                           random_state=RANDOM_SEED)
    db  = davies_bouldin_score(X_ref_sc, lbl)
    sizes = pd.Series(lbl).value_counts().sort_index().tolist()
    diag_rows.append({"K": k,
                      "Inertia":         round(km.inertia_, 1),
                      "Silhouette":      round(sil, 4),
                      "Davies-Bouldin":  round(db, 4),
                      "Cluster sizes":   sizes})
    print(f"K={k}  Inertia={km.inertia_:.1f}  "
          f"Silhouette={sil:.4f}  DB={db:.4f}  sizes={sizes}")

diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(CLUSTER_DIR / "cluster_diagnostics.csv", index=False)
print(f"\nSaved → {CLUSTER_DIR}/cluster_diagnostics.csv")


In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))

ax1.plot(diag_df["K"], diag_df["Inertia"], "o-", color="steelblue", lw=1.5)
ax1.set_xlabel("K"); ax1.set_ylabel("Inertia"); ax1.set_title("Elbow (inertia)")
ax1.grid(True, alpha=0.3)

ax2.plot(diag_df["K"], diag_df["Silhouette"], "o-", color="firebrick", lw=1.5)
ax2.set_xlabel("K"); ax2.set_ylabel("Silhouette score"); ax2.set_title("Silhouette ↑")
ax2.grid(True, alpha=0.3)

ax3.plot(diag_df["K"], diag_df["Davies-Bouldin"], "o-", color="seagreen", lw=1.5)
ax3.set_xlabel("K"); ax3.set_ylabel("Davies-Bouldin"); ax3.set_title("Davies-Bouldin ↓")
ax3.grid(True, alpha=0.3)

fig.suptitle(f"K-Means diagnostics — {len(obsmodel)}-feature obsmodel", fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "cluster_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR}/cluster_diagnostics.png")


## 5. Apply cluster labels to Greenland & Antarctica grids

`kmeans.predict()` uses the same centroids fitted on the global reference, so
cluster semantics are identical in both domains.  Features not present in a
target grid are flagged as missing; NaN rows receive label **−1**.


In [ ]:
DOMAINS = {
    "Greenland":  GREENLAND_PARQUET,
    "Antarctica": ANTARCTICA_PARQUET,
}

target_labels = {}   # domain -> {k: label_array}

for domain, parquet_path in DOMAINS.items():
    print(f"\n── {domain} ──")
    df_tgt = pd.read_parquet(parquet_path)
    print(f"  Grid rows: {len(df_tgt):,}")

    # Check feature availability
    missing = [f for f in obsmodel if f not in df_tgt.columns]
    if missing:
        print(f"  WARNING: {len(missing)} features missing from parquet — "
              f"will be filled with NaN: {missing}")
        for m in missing:
            df_tgt[m] = np.nan

    X_tgt_raw = df_tgt[obsmodel].replace([np.inf, -np.inf], np.nan).values.astype(np.float32)

    # Mask rows that have ANY NaN in the feature vector
    valid_mask = np.isfinite(X_tgt_raw).all(axis=1)
    print(f"  Valid rows (no NaN): {valid_mask.sum():,} / {len(X_tgt_raw):,}")

    X_tgt_sc = np.full_like(X_tgt_raw, np.nan)
    X_tgt_sc[valid_mask] = scaler.transform(X_tgt_raw[valid_mask])

    target_labels[domain] = {}
    for k in K_VALUES:
        km  = kmeans_models[k]
        lbl = np.full(len(X_tgt_raw), -1, dtype=np.int16)  # -1 = NaN rows
        lbl[valid_mask] = km.predict(X_tgt_sc[valid_mask]).astype(np.int16)
        target_labels[domain][k] = lbl
        counts = pd.Series(lbl[valid_mask]).value_counts().sort_index().to_dict()
        print(f"  K={k}  cluster sizes: {counts}")

    # Attach labels to dataframe and save
    for k in K_VALUES:
        df_tgt[f"cluster_k{k}"] = target_labels[domain][k]

    out_path = CLUSTER_DIR / f"{domain.lower()}_cluster_labels.parquet"
    df_tgt[[c for c in df_tgt.columns if c in ["lon", "lat"] or c.startswith("cluster_")]].to_parquet(out_path, index=False)
    print(f"  Saved cluster labels → {out_path}")


## 6. Cluster maps

In [ ]:
CMAPS = {3: "Set1", 4: "Set2", 5: "tab10"}

for domain in DOMAINS:
    df_lbl = pd.read_parquet(CLUSTER_DIR / f"{domain.lower()}_cluster_labels.parquet")

    fig, axes = plt.subplots(1, len(K_VALUES), figsize=(6 * len(K_VALUES), 5))
    fig.suptitle(f"{domain} — K-Means cluster maps", fontsize=13)

    for ax, k in zip(axes, K_VALUES):
        col = f"cluster_k{k}"
        valid = df_lbl[col] >= 0
        sc = ax.scatter(
            df_lbl.loc[valid, "lon"],
            df_lbl.loc[valid, "lat"],
            c=df_lbl.loc[valid, col],
            cmap=CMAPS[k], s=0.5, linewidths=0,
            vmin=-0.5, vmax=k - 0.5
        )
        ax.set_title(f"K={k}")
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        fig.colorbar(sc, ax=ax, ticks=range(k), label="Cluster")

    fig.tight_layout()
    fig.savefig(FIG_DIR / f"clusters_{domain.lower()}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {FIG_DIR}/clusters_{domain.lower()}.png")


In [ ]:
# Centroid heatmap — feature means per cluster (K=best, usually 4 or 5)
# Shows the geophysical "fingerprint" of each cluster.
fig, axes = plt.subplots(len(K_VALUES), 1, figsize=(14, 4 * len(K_VALUES)))
if len(K_VALUES) == 1:
    axes = [axes]

for ax, k in zip(axes, K_VALUES):
    km = kmeans_models[k]
    # Centroids are in standardised space; invert to physical units
    centres_phys = scaler.inverse_transform(km.cluster_centers_)
    df_centres = pd.DataFrame(centres_phys, columns=obsmodel)

    # Normalise each feature to [0, 1] for display
    df_norm = (df_centres - df_centres.min()) / (df_centres.max() - df_centres.min() + 1e-12)

    im = ax.imshow(df_norm.values, aspect="auto", cmap="RdYlBu_r", vmin=0, vmax=1)
    ax.set_xticks(range(len(obsmodel)))
    ax.set_xticklabels(obsmodel, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(k))
    ax.set_yticklabels([f"Cluster {i}" for i in range(k)])
    ax.set_title(f"K={k} centroid profiles (normalised physical units)", fontsize=10)
    plt.colorbar(im, ax=ax, label="Normalised value")

fig.tight_layout()
fig.savefig(FIG_DIR / "cluster_centroid_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {FIG_DIR}/cluster_centroid_profiles.png")


## 7. Save cluster summary JSON

In [ ]:
summary = {
    "notebook":      "6_CLUSTERING",
    "obsmodel":      obsmodel,
    "n_features":    len(obsmodel),
    "K_values":      K_VALUES,
    "random_seed":   RANDOM_SEED,
    "note":          (
        "KMeans fitted on global IHFC reference (no spatial coords). "
        "Same centroids applied to Greenland and Antarctica grids. "
        "NaN grid cells receive label -1."
    ),
    "diagnostics":   diag_rows,
}
out_json = CLUSTER_DIR / "cluster_summary.json"
with open(out_json, "w") as fp:
    json.dump(summary, fp, indent=2)
print(f"Saved → {out_json}")
print(f"\nDiagnostics table:")
print(diag_df.to_string(index=False))
